# Final Task B 17-class - ConvNeXt-Tiny

Train ConvNeXt-Tiny directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [2]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17-new"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17-new: {FINAL_B_DIR}")


ROOT: /workspace/woodbench
task-B-17-new: /workspace/woodbench/notebooks/phase 1/task-B-17-new


In [3]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    %pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [4]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "convnext_tiny"
DISPLAY_NAME = "ConvNeXt-Tiny"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: ConvNeXt-Tiny (convnext_tiny)
Batch size: 32
Data root: /workspace/woodbench/data/task-B-group-split


In [5]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [6]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-new/final

main/seed_42/convnext_tiny
[main | convnext_tiny s42] train=3365, val=903, test=532, classes=17


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 3.2412, Acc: 2.86% | Val Loss: 3.2064, Acc: 5.20% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 5.20%)


Epoch 2/50 | Train Loss: 1.1069, Acc: 83.01% | Val Loss: 0.7182, Acc: 96.46% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 96.46%)


Epoch 3/50 | Train Loss: 0.6656, Acc: 98.27% | Val Loss: 0.7226, Acc: 94.13% | LR: 6.00e-05[EarlyStop 1/10]


Epoch 4/50 | Train Loss: 0.6464, Acc: 98.27% | Val Loss: 0.6386, Acc: 98.01% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 98.01%)


... (161 outputs left out)

Epoch 23/50 | Train Loss: 1.0065, Acc: 100.00% | Val Loss: 1.0123, Acc: 99.67% | LR: 6.55e-05[EarlyStop 8/10]


Epoch 24/50 | Train Loss: 1.0080, Acc: 99.91% | Val Loss: 1.0181, Acc: 99.45% | LR: 6.21e-05[EarlyStop 9/10]


Epoch 25/50 | Train Loss: 1.0066, Acc: 99.97% | Val Loss: 1.0143, Acc: 99.67% | LR: 5.87e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 25.
[ERROR] taskB_label_smoothing_0.2/seed_42/convnext_tiny: RuntimeError('[enforce fail at inline_container.cc:672] . unexpected pos 318965504 vs 318965396')


RuntimeError: convnext_tiny failed runs: {'taskB_label_smoothing_0.2/seed_42/convnext_tiny': "RuntimeError('[enforce fail at inline_container.cc:672] . unexpected pos 318965504 vs 318965396')"}

In [7]:
summarize_results(results)


NameError: name 'results' is not defined

## Continue after disk-full interruption

Run the root, import/config, and data-check cells above, but do not rerun the old `run_all_experiments` cell. The label-smoothing=0.2 run reached early stopping at epoch 25, but its final `last.pth` is corrupt. Finalize directly from the valid epoch-15 `best.pth`; this deliberately ignores the corrupt `last.pth`.

In [8]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [10]:
from final_b_17class import (
    EXPERIMENTS,
    experiment_output_dir,
    load_completed_results,
    train_one,
)

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

def finalize_from_best(experiment_name, seed):
    output_dir = experiment_output_dir(ROOT, experiment_name, seed, BACKBONE)
    best_path = output_dir / f"{BACKBONE}_seed{seed}_best.pth"
    if not best_path.exists():
        raise FileNotFoundError(f"Missing best checkpoint: {best_path}")
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=False,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 5/6 completed configs for convnext_tiny.


In [11]:
# Ignore the corrupt last.pth and evaluate/register the valid best.pth.
finalize_from_best("taskB_label_smoothing_0.2", 42)


[taskB_label_smoothing_0.2 | convnext_tiny s42] Found checkpoint: /workspace/woodbench/data/experiments/Task-B-new/final/taskB_label_smoothing_0.2/convnext_tiny/convnext_tiny_seed42_best.pth
[taskB_label_smoothing_0.2 | convnext_tiny s42] test_acc=99.06% F1=99.14%


{'backbone': 'convnext_tiny',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'best_val_acc': 99.88925802879291,
 'best_epoch': 15,
 'final_train_acc': 99.85119047619048,
 'final_val_acc': 99.88925802879291,
 'stopped_early': False,
 'epochs_trained': 15,
 'test_accuracy': 0.9906015037593985,
 'test_f1_macro': 0.9913731212352876,
 'test_precision_macro': 0.9917339484813534,
 'test_recall_macro': 0.9918300653594772,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecyparis obtusa': 0.9850746268656716,
  'Cryptomeria japonica': 1.0,
  'Cunninghamia lanceolata': 1.0,
  'Fokienia hodginsii': 1.0,
  'Picea glauca': 1.0,
  'Pinus cernua': 1.0,
  'Pinus dalatensis': 0.9411764705882353,
  'Pinus krempfii': 0.9411764705882353,
  'Pinus radiata': 1.0,
  'Podocarpus pilgeri': 0.9859154929577465,
  'Pseudotsuga brevifolia': 1.0,
  'Taxus wallichiana': 1.0,
  'Thujopsis dolabrata': 1.0,
  'Xanthoc

In [12]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    99.89%    99.62%    99.66%    99.65%    99.67%
main                              123    99.89%    99.06%    99.14%    99.16%    99.18%
main                              456   100.00%    99.81%    99.83%    99.82%    99.84%
taskB_90rotation                   42    99.78%    99.81%    99.83%    99.82%    99.84%
taskB_label_smoothing_0.0          42   100.00%    99.62%    99.66%    99.65%    99.67%
taskB_label_smoothing_0.2          42    99.89%    99.06%    99.14%    99.17%    99.18%
----------------------------------------------------------------------------------------
main test_acc: 99.50 +/- 0.32%
main test_f1:  99.54 +/- 0.29%
